# Facility type × import seasonality

Explore how desk **`primary_tag`** (exclusive bucket) and **`tag_*` flags** (multi-label) relate to
**import consumption patterns** — especially whether imports are **seasonal** or steady.

**Inputs**
- `data/manual/terminal_desk_map.csv` — `primary_tag` plus boolean `tag_*` columns (blank = False)
- `data/derived/terminal_product_monthly.parquet` — from notebook `04` (re-run §2+§8 if stale)

**Seasonality metrics** (per Installation × Product, imports only)
- `import_yoy_corr` — same calendar month vs prior year (higher → repeating seasonality)
- `climatology_cv` — spread of the 12-month average profile (higher → winter/summer shape)
- `import_cv` — overall month-to-month volatility (includes irregular spikes)
- `seasonal_label` — coarse bucket: `seasonal` / `flat` / `mixed` / `irregular`

**Two views**
- §§1–5 partition by **`primary_tag`** (each site in one bucket; volumes add up)
- §6 plots by segment: set `SEGMENT_MODE` to **`"tags"`** (multi-label) or **`"primary"`** (exclusive, no double count)

Default scope: **Spain** installations with a `primary_tag` or any tag flag.

In [14]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from facility_seasonality import (
    aggregate_type_product_imports,
    build_import_seasonality,
    build_segment_import_series,
    plot_bafa_seasonality,
    plot_type_product_timeseries,
)
from terminal_desk import (
    DESK_MAP_PATH,
    is_tag_true,
    load_desk_map,
    tag_columns,
    tag_name,
)

DERIVED = ROOT / "data" / "derived"
MONTHLY_PATH = DERIVED / "terminal_product_monthly.parquet"

FOCUS_COUNTRY = "Italy"
MIN_LIFETIME_KB = 500.0  # skip thin installation×product pairs
TOP_PRODUCTS = ["Gasoil", "Gasoline", "FO", "Naphtha", "Jet", "Diesel"]

print("ROOT", ROOT)
assert MONTHLY_PATH.exists(), f"Missing {MONTHLY_PATH} — run notebook 04 through Persist"
assert DESK_MAP_PATH.exists(), f"Missing {DESK_MAP_PATH}"

ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast


In [15]:
desk = load_desk_map(DESK_MAP_PATH)
in_country = desk["country"].eq(FOCUS_COUNTRY)
has_primary = desk["primary_tag"].notna() & desk["primary_tag"].astype(str).str.strip().ne("")
has_tag = False
for col in tag_columns(desk):
    has_tag = has_tag | desk[col].map(is_tag_true)

mapped = desk.loc[in_country & (has_primary | has_tag)].copy()
mapped["primary_tag"] = mapped["primary_tag"].astype(str).str.strip().replace({"nan": ""})
mapped["facility_type"] = mapped["primary_tag"]  # exclusive partition alias for §§1–5

print(f"{FOCUS_COUNTRY} installations tagged:", len(mapped))
print("primary_tag")
print(mapped.loc[mapped["primary_tag"].ne(""), "primary_tag"].value_counts())
print("\ntag flags (TRUE counts)")
for col in tag_columns(desk):
    n = int(mapped[col].map(is_tag_true).sum())
    if n:
        print(f"  {tag_name(col)}: {n}")
mapped.loc[mapped["primary_tag"].ne(""), ["installation_name", "primary_tag", "owner", "zone"]].head(10)

Italy installations tagged: 65
primary_tag
primary_tag
storage           28
refinery          11
petchem            9
agriculture        4
biofuels           4
bunkering          3
airport            2
military           1
biofuel            1
power_plant        1
heavy_industry     1
Name: count, dtype: int64

tag flags (TRUE counts)
  refinery: 11
  storage: 29
  blending: 25
  bunkering: 4
  heavy_industry: 1
  power_plant: 1
  airport: 2
  agriculture: 4
  military: 1
  biofuels: 8
  petchem: 9


,installation_name,primary_tag,owner,zone
36,Sarroch Refinery,refinery,NaN,Sarroch
95,Milazzo Refinery,refinery,ENI,Milazzo
105,Sonatrach Augusta Refinery,refinery,Sonatrach,Augusta
109,ISAB,refinery,Ludoil,Augusta
154,ISAB South,refinery,Ludoil,Augusta
180,Naples Refinery,storage,NaN,Naples
210,Genoa,refinery,IPLOM,Genoa
220,Fiumicino,airport,Italiana Petroli,Fiumicino
261,Venice,storage,NaN,Venice
335,Leghorn Livorno,storage,ENI,Livorno


In [16]:
monthly = pd.read_parquet(MONTHLY_PATH)
monthly["year_month"] = pd.to_datetime(monthly["year_month"])

inst_types = (
    mapped.loc[mapped["primary_tag"].ne("")]
    .set_index("installation_name")["primary_tag"]
    .to_dict()
)
monthly = monthly.loc[
    monthly["Country"].eq(FOCUS_COUNTRY)
    & monthly["Installation"].isin(inst_types)
].copy()
monthly["primary_tag"] = monthly["Installation"].map(inst_types)
monthly["facility_type"] = monthly["primary_tag"]
monthly["cal_month"] = monthly["year_month"].dt.month

imports = monthly.loc[monthly["Imp/Exp"].eq("Import")].copy()
print(f"import monthly rows (mapped {FOCUS_COUNTRY}): {len(imports):,}")
imports.groupby("primary_tag")["kb_abs"].sum().sort_values(ascending=False)

import monthly rows (mapped Italy): 8,102


primary_tag
storage        911321.131
refinery       348268.668
airport        255523.520
petchem         73416.989
biofuel         72642.613
military        18824.464
bunkering       16458.880
biofuels         5730.489
power_plant       291.386
agriculture       227.599
Name: kb_abs, dtype: float64

## 1) Seasonality scorecard by installation × product

In [17]:
seasonality = build_import_seasonality(imports)
seasonality = seasonality.loc[seasonality["total_import_kb"] >= MIN_LIFETIME_KB].copy()
seasonality = seasonality.merge(
    mapped[["installation_name", "primary_tag", "facility_type", "owner", "zone"]],
    left_on="Installation",
    right_on="installation_name",
    how="left",
)

print(f"scored pairs: {len(seasonality):,}")
print(seasonality["seasonal_label"].value_counts())
seasonality.sort_values("total_import_kb", ascending=False).head(12)

scored pairs: 193
seasonal_label
irregular               121
mixed                    55
seasonal                  8
flat                      7
insufficient_history      2
Name: count, dtype: int64


,Installation,Product,n_months,total_import_kb,mean_import_kbd,import_cv,import_yoy_corr,climatology_cv,seasonal_label,installation_name,primary_tag,facility_type,owner,zone
178,Venice,Diesel,115,100616.409,28.687200,0.503146,-0.105216,0.208558,flat,Venice,storage,storage,NaN,Venice
103,Naples Refinery,Gasoil,115,82825.094,23.721450,0.576059,0.250107,0.176630,mixed,Naples Refinery,storage,storage,NaN,Naples
31,Fiumicino,Jet,114,77284.606,22.268931,0.745777,0.218658,0.215235,mixed,Fiumicino,airport,airport,Italiana Petroli,Fiumicino
28,Fiumicino,Diesel,114,73052.335,21.024810,0.651352,-0.177183,0.182372,flat,Fiumicino,airport,airport,Italiana Petroli,Fiumicino
101,Naples Refinery,Diesel,115,66854.993,19.050093,0.677665,0.098042,0.247657,mixed,Naples Refinery,storage,storage,NaN,Naples
180,Venice,Gasoil,116,63454.530,18.007415,0.633134,-0.105544,0.219533,flat,Venice,storage,storage,NaN,Venice
104,Naples Refinery,Gasoline,116,57753.119,16.345294,0.502006,0.333606,0.114126,mixed,Naples Refinery,storage,storage,NaN,Naples
20,ENI Porto Marghera Refinery,Naphtha,106,44138.884,13.683016,0.620458,0.025088,0.176390,mixed,ENI Porto Marghera Refinery,biofuel,biofuel,ENI,Marghera Port
117,Petra,Gasoil,111,35537.701,10.505875,0.536383,-0.094743,0.150980,flat,Petra,storage,storage,Gruppo PIRP,Ravenna
192,Versalis Cracker Brindisi,Naphtha,98,30254.526,10.127827,0.598371,0.027904,0.212724,mixed,Versalis Cracker Brindisi,petchem,petchem,NaN,Brindisi


## 2) Roll up by `primary_tag`

Volume-weighted view of the **exclusive** bucket. Larger import streams count more in the averages.

In [18]:
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    m = values.notna() & weights.notna() & weights.gt(0)
    if not m.any():
        return np.nan
    return float(np.average(values[m], weights=weights[m]))


type_summary = (
    seasonality.groupby("facility_type", dropna=False)
    .apply(
        lambda g: pd.Series(
            {
                "n_pairs": len(g),
                "total_import_kb": g["total_import_kb"].sum(),
                "yoy_corr_wavg": weighted_mean(g["import_yoy_corr"], g["total_import_kb"]),
                "climatology_cv_wavg": weighted_mean(g["climatology_cv"], g["total_import_kb"]),
                "import_cv_wavg": weighted_mean(g["import_cv"], g["total_import_kb"]),
                "pct_seasonal": (g["seasonal_label"].eq("seasonal")).mean(),
                "pct_flat": (g["seasonal_label"].eq("flat")).mean(),
            }
        ),
        include_groups=True,
    )
    .reset_index(level=0)
    .sort_values("total_import_kb", ascending=False)
)
type_summary

C:\Users\luiscarlos.gaitan\AppData\Local\Temp\ipykernel_56448\657111146.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,facility_type,n_pairs,total_import_kb,yoy_corr_wavg,climatology_cv_wavg,import_cv_wavg,pct_seasonal,pct_flat
7,storage,84.0,901757.615,0.060636,0.301145,0.962713,0.083333,0.059524
6,refinery,63.0,343309.345,-0.063921,0.548973,1.719515,0.000000,0.000000
0,airport,9.0,253990.547,-0.013856,0.271269,0.849543,0.000000,0.222222
1,biofuel,7.0,72206.750,-0.021105,0.290320,0.986478,0.000000,0.000000
5,petchem,17.0,69894.990,0.037230,0.442627,1.231232,0.058824,0.000000
4,military,5.0,18772.077,-0.076887,0.544172,1.702057,0.000000,0.000000
3,bunkering,6.0,14347.924,-0.069720,0.612464,1.899241,0.000000,0.000000
2,biofuels,2.0,3491.903,-0.017469,0.854617,1.950119,0.000000,0.000000


In [19]:
fig = px.box(
    seasonality,
    x="facility_type",
    y="import_yoy_corr",
    color="facility_type",
    points="outliers",
    title=f"Import YoY month correlation by facility_type ({FOCUS_COUNTRY})",
    labels={"import_yoy_corr": "YoY same-month correlation (higher = more seasonal)"},
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(showlegend=False, height=480)
fig.show()

fig2 = px.box(
    seasonality,
    x="facility_type",
    y="climatology_cv",
    color="facility_type",
    points="outliers",
    title="Calendar profile spread (climatology CV)",
)
fig2.update_layout(showlegend=False, height=480)
fig2.show()

## 3) Calendar-month import profile by facility type

Normalized to each type's annual average (1.0 = average month).

In [20]:
prod_imports = imports.loc[imports["Product"].isin(TOP_PRODUCTS)].copy()

type_month = (
    prod_imports.groupby(["facility_type", "cal_month"], as_index=False)["kbd"]
    .sum()
    .sort_values(["facility_type", "cal_month"])
)
type_month["month_name"] = pd.to_datetime(type_month["cal_month"], format="%m").dt.strftime("%b")
type_month["season_index"] = type_month.groupby("facility_type")["kbd"].transform(
    lambda s: s / s.mean() if s.mean() > 0 else np.nan
)

fig = px.line(
    type_month,
    x="month_name",
    y="season_index",
    color="facility_type",
    markers=True,
    category_orders={"month_name": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]},
    title=f"Seasonal import shape by facility_type ({', '.join(TOP_PRODUCTS[:4])}…)",
    labels={"season_index": "vs type annual avg (=1)"},
)
fig.add_hline(y=1.0, line_dash="dot", line_color="gray")
fig.update_layout(height=520)
fig.show()

px.imshow(
    type_month.pivot(index="facility_type", columns="month_name", values="season_index"),
    aspect="auto",
    color_continuous_scale="RdYlBu_r",
    title="Season index heatmap (facility_type × month)",
    labels=dict(color="vs avg"),
).update_layout(height=420)

## 4) Product × facility_type detail

Which products drive seasonality within each type?

In [21]:
def _agg_prod_type(g: pd.DataFrame) -> pd.Series:
    w = g["total_import_kb"]
    return pd.Series(
        {
            "total_import_kb": w.sum(),
            "yoy_corr": weighted_mean(g["import_yoy_corr"], w),
            "clim_cv": weighted_mean(g["climatology_cv"], w),
            "pct_seasonal": (g["seasonal_label"] == "seasonal").mean(),
        }
    )


prod_type = (
    seasonality.loc[seasonality["Product"].isin(TOP_PRODUCTS)]
    .groupby(["facility_type", "Product"], as_index=False)
    .apply(_agg_prod_type, include_groups=True)
    .reset_index(drop=True)
    .sort_values(["facility_type", "total_import_kb"], ascending=[True, False])
)
prod_type

C:\Users\luiscarlos.gaitan\AppData\Local\Temp\ipykernel_56448\2237763374.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_prod_type, include_groups=True)


,facility_type,Product,total_import_kb,yoy_corr,clim_cv,pct_seasonal
3,airport,Jet,96365.998,0.179845,0.229886,0.000000
0,airport,Diesel,82768.597,-0.178067,0.211233,0.000000
2,airport,Gasoline,37905.531,-0.023956,0.212776,0.000000
1,airport,Gasoil,35506.454,-0.146797,0.482978,0.000000
4,airport,Naphtha,1443.967,0.005866,2.804071,0.000000
10,biofuel,Naphtha,44138.884,0.025088,0.176390,0.000000
7,biofuel,Gasoil,8509.588,-0.053383,0.349961,0.000000
8,biofuel,Gasoline,6488.330,-0.061848,0.491732,0.000000
9,biofuel,Jet,5885.378,-0.092565,0.394398,0.000000
6,biofuel,FO,2271.396,-0.333333,0.807054,0.000000


In [22]:
heat = prod_type.pivot(index="facility_type", columns="Product", values="yoy_corr")
fig = px.imshow(
    heat.loc[:, [p for p in TOP_PRODUCTS if p in heat.columns]],
    aspect="auto",
    color_continuous_scale="RdBu",
    zmin=-0.5,
    zmax=0.5,
    title="YoY seasonality by facility_type × product",
    labels=dict(color="import_yoy_corr"),
)
fig.update_layout(height=440)
fig.show()

## 5) Example time series

Pick a `facility_type` + `Product` below to inspect raw monthly import kbd.

In [23]:
EXAMPLE_TYPE = "refinery"
EXAMPLE_PRODUCT = "Naphtha"

examples = seasonality.loc[
    seasonality["facility_type"].eq(EXAMPLE_TYPE)
    & seasonality["Product"].eq(EXAMPLE_PRODUCT)
].sort_values("total_import_kb", ascending=False)
print(examples[["Installation", "seasonal_label", "import_yoy_corr", "climatology_cv", "total_import_kb"]].head(8))

if examples.empty:
    print("No examples for this type/product — change EXAMPLE_TYPE / EXAMPLE_PRODUCT")
else:
    top_inst = examples.iloc[0]["Installation"]
    ts = imports.loc[
        imports["Installation"].eq(top_inst) & imports["Product"].eq(EXAMPLE_PRODUCT)
    ].sort_values("year_month")
    fig = px.bar(
        ts,
        x="year_month",
        y="kbd",
        title=f"{top_inst} — {EXAMPLE_PRODUCT} imports (kbd)",
        labels={"kbd": "kbd"},
    )
    fig.update_layout(height=420)
    fig.show()

         Installation seasonal_label  import_yoy_corr  climatology_cv  \
55               ISAB          mixed         0.035432        0.187467   
141  Sarroch Refinery      irregular         0.001930        0.558814   
47              Genoa      irregular        -0.138319        0.635112   
164           Taranto      irregular         0.055703        0.800433   
134     Sarroch Crude      irregular        -0.200000        1.531678   

     total_import_kb  
55         24374.690  
141         8566.919  
47          3791.461  
164         1192.852  
134          915.209  


## 6) Time series & BAFA seasonality (segment × product)

Aggregated **import kbd** by segment. Flip `SEGMENT_MODE` in the next cell:

- **`"tags"`** — multi-label `tag_*` membership (a storage+bunkering site appears in both; do not sum segments)
- **`"primary"`** — exclusive `primary_tag` (each site once; volumes add up)

Plots:

- **Time series** — one figure per segment, products as subplots, shared date axis
- **BAFA-style seasonality** — Jan–Dec on the x-axis, one line per calendar year (current year **bold red**)

Adjust `FOCUS_SEGMENTS` / `PLOT_PRODUCTS` below. Pairs below `MIN_PAIR_KB` lifetime imports are skipped.

In [24]:
# "tags" = multi-label tag_* (can double-count); "primary" = exclusive primary_tag
SEGMENT_MODE = "primary"  # or "primary"
FOCUS_SEGMENTS = [
    "storage",
    "refinery",
    "bunkering",
    "heavy_industry",
    "blending",
    "military",
    "petchem"
]
PLOT_PRODUCTS = TOP_PRODUCTS
MIN_PAIR_KB = 5_000.0

seg = build_segment_import_series(
    imports,
    mapped,
    mode=SEGMENT_MODE,
    focus=FOCUS_SEGMENTS,
    products=PLOT_PRODUCTS,
    min_pair_kb=MIN_PAIR_KB,
)
type_product_ts = seg["type_product_ts"]
active_pairs = seg["active_pairs"]
active_tags = seg["active_tags"]
SEGMENT_LABEL = seg["label"]
CURRENT_YEAR = seg["current_year"]

print(f"mode={SEGMENT_MODE!r} ({seg['note']})")
print(f"active {SEGMENT_LABEL}×product pairs (>={MIN_PAIR_KB:,.0f} kb): {len(active_pairs)}")
active_pairs.sort_values("total_kb", ascending=False)

mode='primary' (each site counted once under its primary_tag)
active primary×product pairs (>=5,000 kb): 17


,segment,Product,total_kb
25,storage,Gasoil,358089.015
23,storage,Diesel,271032.983
26,storage,Gasoline,153562.428
19,refinery,Gasoil,65224.908
27,storage,Jet,46278.309
17,refinery,Diesel,40926.410
22,refinery,Naphtha,39891.206
24,storage,FO,34127.787
16,petchem,Naphtha,32137.331
18,refinery,FO,28938.994


In [25]:
for tag in active_tags:
    prods = active_pairs.loc[active_pairs["segment"].eq(tag), "Product"].tolist()
    try:
        fig_ts = plot_type_product_timeseries(
            type_product_ts,
            segment=tag,
            products=prods,
            title=f"{FOCUS_COUNTRY} — {SEGMENT_LABEL}:{tag} imports — monthly kbd",
        )
    except ValueError as exc:
        print(exc)
        continue
    fig_ts.show()

### BAFA seasonality grids (one figure per segment)

Each subplot is a **product**; lines are **calendar years**; **red** = `CURRENT_YEAR`.
Titles use `SEGMENT_LABEL` (`tag` or `primary`) from the setup cell.

In [26]:
for tag in active_tags:
    prods = (
        active_pairs.loc[active_pairs["segment"].eq(tag), "Product"]
        .tolist()
    )
    if not prods:
        continue
    fig_bafa = plot_bafa_seasonality(
        type_product_ts,
        segment=tag,
        products=prods,
        current_year=CURRENT_YEAR,
        title=f"{FOCUS_COUNTRY} — {SEGMENT_LABEL}:{tag} imports — seasonality (kbd)",
    )
    fig_bafa.show()

## 7) Takeaways to validate manually

- **Refineries** — often steady feedstock imports (`flat` / low climatology CV) except turnaround windows.
- **Port storage / bunkering** — may show summer/winter tourism or heating-oil shapes on FO/Gasoil.
- **Heavy industry** — can be tied to plant operating schedules (check Aviles / fertilizer sites).
- Negative `import_yoy_corr` often means **regime changes** or one-off spikes, not clean seasonality.

Re-run notebook `04` Persist if monthly data is stale after a Kpler refresh.